#### Lab Exercise 3

**Naive Bayes Classifier**  

**Maximum marks = 30**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import re, os
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In this exercise you will build a Naive Bayes (NB) Classifier. The functions developed in Lab Exercise 2 will prove useful.

In [2]:
def prepare_data(fl, x):
    """
    Reads the dataset and splits it into training and test sets.

    Args:
      fl (str): filename
      x (float): fraction of data to be used for training i.e 0.8 for 80% train

    Returns:
      df_tr (dataframe): training data
      df_ts (dataframe): test data
    """
    df = pd.read_csv(fl)

    # Shuffle the data
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    train_size = int(len(df) * x)
    df_tr = df.iloc[:train_size].reset_index(drop=True) #dataframe for training data
    df_ts = df.iloc[train_size:].reset_index(drop=True) #dataframe for test data
    return df_tr, df_ts

In [3]:
# This is a list of stopwords from NLTK.
from nltk.corpus import stopwords
stop_words = stopwords.words('english') # list of stopwords
neg_words = ["no", "not","don't", "nor"]
# This is a list of "negative" words. You may modify it, but do it only with words in "stop_words" list.
# Add code to keep negative words even if they are stop words
stop_words = [word for word in stop_words if word not in neg_words]

Create a separate tokenization function.

In [4]:
def tokenize(text):
  """
  Convert text into a list of cleaned tokens.
  Transforms text into lowercase, extract words, remove stopwords.

  Args:
    text (str): text to be tokenized

  Returns:
    tokens (list): list of tokens
  """
  text = text.lower()
  tokens = re.findall(r"\b[a-z']+\b", text)

  filtered_tokens = [
      token for token in tokens if token not in stop_words
  ]
  return filtered_tokens

Call prepare_dataset function to create train and test sets.

In [5]:
fl = "Emotion_classify_Data.csv"
df_train, df_test = prepare_data(fl, 0.8)

**Task 1**  
*Complete the function "build_vocb_freq" below.*  **[10 marks]**
1. Build the vocabulary. The task is the same as in Lab Exercise 2, but now you remove the stopwords from vocabulary, except the negative words in the *neg_words* list.
2. The output is dictionary. The keys in the dictionary are the word tokens. The values are three frequencies.
3. For example, if the word is "*sad*", then estimate $P(\text{'sad'}|class=\text{'anger'}), \; P(\text{'sad'}|class=\text{'fear'}), \; P(\text{'sad'}|class=\text{'joy'})$. So the values are 3-tuple of numbers. Remember the ordering.
4. Use *defaultdict* with default value 1. We choose 1, because when we take logarithms $\log 1 = 0$.


In [6]:
# The output of this function is a dictionary.
def build_vocb_freq(df):
  """
  Build a vocabulary dictionary whereby:
    key   -> token
    value -> (P(token|anger), P(token|fear), P(token|joy))

  Args:
    df (dataframe): training data

  Returns:
    vocb_freq (dict): dictionary
  """
  classes = ['anger', 'fear', 'joy']

  # Defaultdict to count token occurence per class, with add-one smoothing
  token_counts = defaultdict(lambda:[1, 1, 1])

  # Total token counts per class, initialized for smoothing
  total_counts = {cls: 0 for cls in classes}

  class_to_idx = {cls: i for i, cls in enumerate(classes)}

  for _, row in df.iterrows():
    text = row['Comment']
    label = row['Emotion']

    tokens = tokenize(text)

    for token in tokens:
      idx = class_to_idx[label]
      token_counts[token][idx] += 1
      total_counts[label] += 1

  # Vocabulary size for smoothing
  vocab_size = len(token_counts)

  # Convert counts into conditional probabilities
  vocb_freq = {}

  for token, counts in token_counts.items():
    probs = []
    for cls in classes:
      idx = class_to_idx[cls]
      prob = counts[idx] / (total_counts[cls] + vocab_size)
      probs.append(prob)
    vocb_freq[token] = tuple(probs)

  return vocb_freq

**Task 2**    
*Complete the function "naive_bayes" below.*  **[20 marks]**
1. Use the training data (*df_train* above) to calculate the frequencies.
2. Use the formulas from Lab 6 to estimate $$ P(c = i|d) = P(c = i|f_1, f_2, \dotsc, f_k), \; 0 \leq i \leq m-1$$. Here $m=3$.
3. Each document '*d*' is message and the features are word tokens.
4. For your convenience, the formulas for NB classifies are reproduced from Lab 6 notebook.
5. Use logarithms (use *numpy* library function *log* or *log*2 (base 2)).
6. Find the frequencies of each token in input document (use the dictionary from the *build_vocb_freq* function), for each class separately. Take log and sum. Add log of prior class probabilities. This is quite straightforward. For example the class "*joy*" is simply the fraction of documents in the training data whose *emotion* class is *joy*.

In [7]:
# Build the vocabulary once
vocb = build_vocb_freq(df_train)

In [8]:
# Compute class priors
classes = ['anger', 'fear', 'joy']

class_priors = (
    df_train['Emotion']
    .value_counts(normalize=True)
    .reindex(classes, fill_value=0)
    .to_dict()
)

#### Naive Bayes classifier 1
1. Suppose we extract features $f_1, f_2, \dotsc, f_k$ from each document. This means that each document is characterised by its features.
2. Also suppose that each document can be classified into one of the $m$ classes. Let us denote them $0,1, 2, \dotsc, m-1$.
3. Our problem is to find the class to which an unclassified document $d$ belongs.
$$ P(c = i|d) = P(c = i|f_1, f_2, \dotsc, f_k), \; 0 \leq i \leq m-1$$
where $f_1, f_2, \dotsc, f_k$ are the features of $d$.
4. From Bayes theorem
$$  P(c|f_1, f_2, \dotsc, f_k) \propto  P(f_1, f_2, \dotsc, f_k|c)P(c) \text{ (proportional to)}$$
The proportionality constant is not relevant.

#### Naive Bayes classifier 2
1. So we have to compute $P(c)$ and $ P(f_1, f_2, \dotsc, f_k|c)$.
2. Computing $P(c)$ is simple. We use the training set and the fraction of those in class $c$ is taken as $P(c)$. For example, if $m=3$ (three classes) and about 25% are in class 1. Then $P(C=1) = .25$.
3. To compute $ P(f_1, f_2, \dotsc, f_k|c)$ we make the extra assumption that for each document the features are conditionally independent (the naive part!). This implies that
$$  P(f_1, f_2, \dotsc, f_k|c) = P(f_1|c)P(f_2|c) \dotsb P(f_k|C)$$
4. So the problem reduces to calculating $P(f_i |c)$, given a class $c$ what is the probability that a feature is present.
5. With *add-one* smoothing
$$ P(f_i|c) = \frac{\mathrm{count}(f_i, c) +1 } { \sum_i \mathrm{count}(f_i, c) + k} $$
6. Use the formula
   $$ \hat{c} = \underset{c}{\mathrm{argmax}} P(c|d) = \underset{c}{\mathrm{argmax}}(\log{(P(c))} + \sum_i \log{P(f_i|c))} $$ $P(c)$ is the prior probability of class $c$.
8. The class decided by the classifier is $\hat{c}$.
9. Comment your code.

In [9]:
def naive_bayes(doc):
    """
    Classifies a document(i.e comment) into one of the three classes.

    Args:
      doc (str): document to be classified

    Returns:
      cl (str): class of the document doc
    """
    # cl = None #class of the document doc
# create the dictionary in the function "build_vocb_freq" outside and call to estimate the probabilities.
    classes = ['anger', 'fear', 'joy']
    class_to_idx = {cls: i for i, cls in enumerate(classes)}

    tokens = tokenize(doc)

    # Starting with log priors
    scores = {}

    for cls in classes:
      if class_priors[cls] > 0:
        scores[cls] = np.log(class_priors[cls])
      else:
        scores[cls] = float('-inf')

    for token in tokens:
      if token in vocb:
        probs = vocb[token]
        for cls in classes:
          idx = class_to_idx[cls]
          scores[cls] += np.log(probs[idx])

    cl = max(scores, key=scores.get)
    return cl

In [10]:
# The following function is a simple test of accuracy of the model.
# It measures the fraction of documents that the model classified correctly.
def test_acc(df):
    # df contains the data for testing, test with both df_train and df_test
    size = len(df)
    correct = 0
    for i in range(size):
        if naive_bayes(df.iloc[i,0])==df.iloc[i,1]:
            correct +=1
    return correct/size

In [11]:
print("Training accuracy:", test_acc(df_train))
print("Test accuracy:", test_acc(df_test))

Training accuracy: 0.9863129079806275
Test accuracy: 0.9006734006734006
